In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ============================================================
# 1. SETTINGS
# ============================================================

n_estimators = 300
contamination = "auto"
max_samples = "auto"
max_features = 1.0
bootstrap = False

threshold_list = [0.065, 0.08]

train_cycle_fraction = 0.80

total_normal_samples = 50000
total_anomaly_samples = 10000

train_normal_samples = int(total_normal_samples * train_cycle_fraction)
test_normal_samples = total_normal_samples - train_normal_samples
test_anomaly_samples = total_anomaly_samples

n_runs = 5
seeds = [42, 43, 44, 45, 46]

data_path = Path("../data_processed/NASA/nasa_battery_features_rolling.csv")
results_root = Path("../results/iForest_NASA_random_files_5runs")
results_root.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. LOAD DATA
# ============================================================

df = pd.read_csv(data_path)

feature_cols = [
    "V", "I", "T",
    "Current_load", "Voltage_load", "Time",
    "V_diff1", "I_diff1", "T_diff1",
    "V_roll_mean", "V_roll_std", "V_roll_min", "V_roll_max",
    "I_roll_mean", "I_roll_std",
    "T_roll_mean", "T_roll_std"
]

required_cols = ["source_file", "y_true_file"] + feature_cols
df = df.dropna(subset=required_cols).reset_index(drop=True)

df["source_file_num"] = (
    df["source_file"]
    .astype(str)
    .str.extract(r"(\d+)", expand=False)
    .astype(int)
)

df = df.sort_values(["source_file_num", "Time"]).reset_index(drop=True)

file_info = (
    df.groupby("source_file", as_index=False)
      .agg(
          source_file_num=("source_file_num", "first"),
          y_true_file=("y_true_file", "first"),
          n_rows=("source_file", "size")
      )
      .sort_values("source_file_num")
      .reset_index(drop=True)
)

normal_files_info = file_info[file_info["y_true_file"] == 0].reset_index(drop=True)
anomaly_files_info = file_info[file_info["y_true_file"] == 1].reset_index(drop=True)

print("Normal files total:", len(normal_files_info))
print("Anomaly files total:", len(anomaly_files_info))
print("Train normal samples:", train_normal_samples)
print("Test normal samples:", test_normal_samples)
print("Test anomaly samples:", test_anomaly_samples)

# ============================================================
# 3. HELPER
# ============================================================

def safe_name(value):
    return str(value).replace(".", "_").replace("-", "minus_")

all_results = []

# ============================================================
# 4. 5 RANDOM RUNS BY SOURCE_FILE
# ============================================================

for run_id, seed in enumerate(seeds, start=1):
    print("\n================================================")
    print(f"RUN {run_id}/{n_runs}, seed = {seed}")
    print("================================================")

    run_dir = results_root / f"run_{run_id}_seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    normal_files_shuffled = normal_files_info.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    split_normal_idx = int(len(normal_files_shuffled) * train_cycle_fraction)

    train_normal_files = normal_files_shuffled.iloc[:split_normal_idx]["source_file"]
    test_normal_files = normal_files_shuffled.iloc[split_normal_idx:]["source_file"]

    anomaly_files_shuffled = anomaly_files_info.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    test_anomaly_files = anomaly_files_shuffled["source_file"]

    train_window = df[df["source_file"].isin(train_normal_files)].copy()
    test_normal_window = df[df["source_file"].isin(test_normal_files)].copy()
    test_anomaly_window = df[df["source_file"].isin(test_anomaly_files)].copy()

    train_normal = (
        train_window[train_window["y_true_file"] == 0]
        .iloc[:train_normal_samples]
        .copy()
    )

    test_normal = (
        test_normal_window[test_normal_window["y_true_file"] == 0]
        .iloc[:test_normal_samples]
        .copy()
    )

    test_anomaly = (
        test_anomaly_window[test_anomaly_window["y_true_file"] == 1]
        .iloc[:test_anomaly_samples]
        .copy()
    )

    test_data = pd.concat(
        [test_normal, test_anomaly],
        ignore_index=True
    )

    X_train_normal_raw = train_normal[feature_cols].copy()
    X_test_raw = test_data[feature_cols].copy()
    y_test = test_data["y_true_file"].astype(int).copy()

    print("Split type: random split by complete source_file cycles")
    print("Train normal files:", train_normal["source_file"].nunique())
    print("Test normal files:", test_normal["source_file"].nunique())
    print("Test anomaly files:", test_anomaly["source_file"].nunique())
    print("Train normal samples:", len(train_normal))
    print("Test normal samples:", len(test_normal))
    print("Test anomaly samples:", len(test_anomaly))
    print("Test total samples:", len(test_data))

    # ========================================================
    # Scaling
    # ========================================================

    scaler = StandardScaler()

    X_train = pd.DataFrame(
        scaler.fit_transform(X_train_normal_raw),
        columns=feature_cols,
        index=X_train_normal_raw.index
    )

    X_test = pd.DataFrame(
        scaler.transform(X_test_raw),
        columns=feature_cols,
        index=X_test_raw.index
    )

    # ========================================================
    # Train fixed Isolation Forest
    # ========================================================

    model = IsolationForest(
        n_estimators=n_estimators,
        contamination=contamination,
        max_samples=max_samples,
        max_features=max_features,
        bootstrap=bootstrap,
        random_state=seed,
        n_jobs=-1
    )

    model.fit(X_train)

    decision_scores = model.decision_function(X_test).ravel()
    score_samples = model.score_samples(X_test).ravel()
    pred_default_raw = model.predict(X_test)
    pred_default = (pred_default_raw == -1).astype(int)

    print("Decision score min:", decision_scores.min())
    print("Decision score max:", decision_scores.max())
    print("Decision score mean:", decision_scores.mean())

    # ========================================================
    # Threshold experiments
    # ========================================================

    for threshold in threshold_list:
        exp_name = (
            f"run_{run_id}_n_estimators_{n_estimators}"
            f"_contamination_{contamination}"
            f"_threshold_{safe_name(threshold)}"
        )

        exp_dir = run_dir / exp_name
        exp_dir.mkdir(parents=True, exist_ok=True)

        y_pred = (decision_scores < threshold).astype(int)

        df_results = test_data.copy()
        df_results["iforest_pred_default"] = pred_default
        df_results["iforest_pred_default_raw"] = pred_default_raw
        df_results["iforest_decision"] = decision_scores
        df_results["iforest_score"] = score_samples
        df_results["threshold"] = threshold
        df_results["is_anomaly"] = y_pred

        cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        accuracy = accuracy_score(y_test, y_pred) * 100
        precision = precision_score(y_test, y_pred, zero_division=0) * 100
        recall = recall_score(y_test, y_pred, zero_division=0) * 100
        f1 = f1_score(y_test, y_pred, zero_division=0) * 100

        result_row = {
            "run_id": run_id,
            "seed": seed,
            "experiment": exp_name,
            "n_estimators": n_estimators,
            "contamination": contamination,
            "max_samples": max_samples,
            "max_features": max_features,
            "bootstrap": bootstrap,
            "threshold": threshold,
            "split_type": "random_source_file_split_5runs",
            "train_cycle_fraction": train_cycle_fraction,
            "total_normal_samples": total_normal_samples,
            "total_anomaly_samples": total_anomaly_samples,
            "train_normal_samples": len(train_normal),
            "test_normal_samples": len(test_normal),
            "test_anomaly_samples": len(test_anomaly),
            "test_total_samples": len(test_data),
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "tp": tp,
            "accuracy_%": round(accuracy, 2),
            "precision_%": round(precision, 2),
            "recall_%": round(recall, 2),
            "f1_%": round(f1, 2)
        }

        all_results.append(result_row)

        df_results.to_csv(exp_dir / "predictions.csv", index=False)

        print("======================================")
        print("Isolation Forest NASA Battery")
        print("======================================")
        print(f"Run      : {run_id}")
        print(f"Seed     : {seed}")
        print(f"Threshold: {threshold}")
        print(f"Accuracy : {accuracy:.2f}%")
        print(f"Precision: {precision:.2f}%")
        print(f"Recall   : {recall:.2f}%")
        print(f"F1-score : {f1:.2f}%")
        print("Confusion Matrix")
        print(cm)

        # ====================================================
        # Metrics table
        # ====================================================

        metrics_df = pd.DataFrame({
            "Metrika": [
                "Presnosť (Accuracy)",
                "Precíznosť (Precision)",
                "Citlivosť (Recall)",
                "F1-skóre"
            ],
            "Hodnota (%)": [
                round(accuracy, 2),
                round(precision, 2),
                round(recall, 2),
                round(f1, 2)
            ]
        })

        fig, ax = plt.subplots(figsize=(6, 2))
        ax.axis("off")

        table = ax.table(
            cellText=metrics_df.values,
            colLabels=metrics_df.columns,
            loc="center",
            cellLoc="center"
        )

        table.auto_set_font_size(False)
        table.set_fontsize(12)
        table.scale(1, 2)

        plt.title(
            f"Výsledné metriky Isolation Forest – NASA\n"
            f"Run = {run_id}, Threshold = {threshold}",
            fontsize=12,
            pad=20
        )

        plt.savefig(exp_dir / "metrics_table.png", dpi=300, bbox_inches="tight")
        plt.close()

        # ====================================================
        # Confusion matrix
        # ====================================================

        plt.figure(figsize=(6, 5))
        plt.imshow(cm, interpolation="nearest")
        plt.title(f"Matica zámen – Run {run_id}, Threshold = {threshold}")
        plt.colorbar()

        classes = ["Normálne", "Anomálie"]
        tick_marks = np.arange(len(classes))

        plt.xticks(tick_marks, classes)
        plt.yticks(tick_marks, classes)

        threshold_cm = cm.max() / 2.0 if cm.max() > 0 else 0.5

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                color = "black" if cm[i, j] > threshold_cm else "white"
                plt.text(j, i, cm[i, j], ha="center", va="center", color=color)

        plt.xlabel("Predikované triedy")
        plt.ylabel("Skutočné triedy")
        plt.tight_layout()
        plt.savefig(exp_dir / "confusion_matrix.png", dpi=300, bbox_inches="tight")
        plt.close()

        # ====================================================
        # Score histogram
        # ====================================================

        plt.figure(figsize=(10, 5))

        plt.hist(
            decision_scores[y_test.values == 0],
            bins=50,
            alpha=0.7,
            label="Normálne"
        )

        plt.hist(
            decision_scores[y_test.values == 1],
            bins=50,
            alpha=0.7,
            label="Anomálie"
        )

        plt.axvline(
            threshold,
            linestyle="--",
            linewidth=2,
            label=f"Threshold = {threshold}"
        )

        plt.title(
            f"Histogram skóre Isolation Forest – NASA\n"
            f"Run = {run_id}, Threshold = {threshold}"
        )
        plt.xlabel("Decision score")
        plt.ylabel("Počet vzoriek")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(exp_dir / "score_histogram.png", dpi=300, bbox_inches="tight")
        plt.close()

# ============================================================
# 5. SAVE SUMMARY AND MEAN RESULTS
# ============================================================

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values(by=["threshold", "run_id"]).reset_index(drop=True)

results_df.to_csv(
    results_root / "summary_results_all_runs_all_thresholds.csv",
    index=False
)

mean_results_df = (
    results_df
    .groupby(["n_estimators", "contamination", "threshold"], as_index=False)
    .agg({
        "accuracy_%": ["mean", "std"],
        "precision_%": ["mean", "std"],
        "recall_%": ["mean", "std"],
        "f1_%": ["mean", "std"],
        "tn": "mean",
        "fp": "mean",
        "fn": "mean",
        "tp": "mean"
    })
)

mean_results_df.columns = [
    "n_estimators", "contamination", "threshold",
    "accuracy_mean_%", "accuracy_std_%",
    "precision_mean_%", "precision_std_%",
    "recall_mean_%", "recall_std_%",
    "f1_mean_%", "f1_std_%",
    "tn_mean", "fp_mean", "fn_mean", "tp_mean"
]

mean_results_df = mean_results_df.sort_values(
    by="threshold",
    ascending=True
).reset_index(drop=True)

mean_results_df.to_csv(
    results_root / "summary_results_mean_5runs_all_thresholds.csv",
    index=False
)

# ============================================================
# 6. FINAL RESULTS FOLDER
# ============================================================

final_dir = results_root / "FINAL_AVERAGE_RESULTS_ALL_THRESHOLDS"
final_dir.mkdir(parents=True, exist_ok=True)

mean_results_df.to_csv(
    final_dir / "mean_metrics_all_thresholds.csv",
    index=False
)

final_table_df = mean_results_df[[
    "threshold",
    "accuracy_mean_%",
    "accuracy_std_%",
    "precision_mean_%",
    "precision_std_%",
    "recall_mean_%",
    "recall_std_%",
    "f1_mean_%",
    "f1_std_%"
]].copy()

final_table_df = final_table_df.round(2)

final_table_df.to_csv(
    final_dir / "final_metrics_table_all_thresholds.csv",
    index=False
)

fig, ax = plt.subplots(figsize=(14, 3))
ax.axis("off")

table = ax.table(
    cellText=final_table_df.values,
    colLabels=final_table_df.columns,
    loc="center",
    cellLoc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.8)

plt.title(
    "Priemerné výsledky Isolation Forest – NASA (5 behov, rôzne thresholdy)",
    fontsize=12,
    pad=20
)

plt.savefig(
    final_dir / "final_metrics_table_all_thresholds.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# ============================================================
# 7. AVERAGE CONFUSION MATRIX FOR EACH THRESHOLD
# ============================================================

for _, row in mean_results_df.iterrows():
    threshold = row["threshold"]

    threshold_dir = final_dir / f"threshold_{safe_name(threshold)}"
    threshold_dir.mkdir(parents=True, exist_ok=True)

    avg_cm = np.array([
        [row["tn_mean"], row["fp_mean"]],
        [row["fn_mean"], row["tp_mean"]]
    ])

    avg_cm_df = pd.DataFrame(
        avg_cm,
        index=["Skutočné normálne", "Skutočné anomálie"],
        columns=["Predikované normálne", "Predikované anomálie"]
    )

    avg_cm_df.to_csv(
        threshold_dir / "average_confusion_matrix.csv"
    )

    plt.figure(figsize=(6, 5))
    plt.imshow(avg_cm, interpolation="nearest")
    plt.title(f"Priemerná matica zámen – Threshold = {threshold}")
    plt.colorbar()

    classes = ["Normálne", "Anomálie"]
    tick_marks = np.arange(len(classes))

    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)

    threshold_cm = avg_cm.max() / 2.0 if avg_cm.max() > 0 else 0.5

    for i in range(avg_cm.shape[0]):
        for j in range(avg_cm.shape[1]):
            color = "black" if avg_cm[i, j] > threshold_cm else "white"
            plt.text(
                j,
                i,
                f"{avg_cm[i, j]:.1f}",
                ha="center",
                va="center",
                color=color
            )

    plt.xlabel("Predikované triedy")
    plt.ylabel("Skutočné triedy")
    plt.tight_layout()

    plt.savefig(
        threshold_dir / "average_confusion_matrix.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    final_metrics_only_mean = pd.DataFrame({
        "Metrika": [
            "Accuracy (%)",
            "Precision (%)",
            "Recall (%)",
            "F1-score (%)"
        ],
        "Hodnota": [
            round(row["accuracy_mean_%"], 2),
            round(row["precision_mean_%"], 2),
            round(row["recall_mean_%"], 2),
            round(row["f1_mean_%"], 2)
        ]
    })

    final_metrics_only_mean.to_csv(
        threshold_dir / "final_metrics_table.csv",
        index=False
    )

    fig, ax = plt.subplots(figsize=(6, 2))
    ax.axis("off")

    table = ax.table(
        cellText=final_metrics_only_mean.values,
        colLabels=final_metrics_only_mean.columns,
        loc="center",
        cellLoc="center"
    )

    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1, 2)

    plt.title(
        f"Priemerné metriky Isolation Forest\nThreshold = {threshold}",
        fontsize=12,
        pad=20
    )

    plt.savefig(
        threshold_dir / "final_metrics_table.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

print("\n==============================")
print("MEAN RESULTS OVER 5 RUNS FOR ALL THRESHOLDS")
print("==============================")
print(mean_results_df)

print("Done")

Normal files total: 2215
Anomaly files total: 517
Train normal samples: 40000
Test normal samples: 10000
Test anomaly samples: 10000

RUN 1/5, seed = 42
Split type: random split by complete source_file cycles
Train normal files: 131
Test normal files: 33
Test anomaly files: 33
Train normal samples: 40000
Test normal samples: 10000
Test anomaly samples: 10000
Test total samples: 20000
Decision score min: -0.2528629619368732
Decision score max: 0.1388522526356562
Decision score mean: -0.007605120383108164
Isolation Forest NASA Battery
Run      : 1
Seed     : 42
Threshold: 0.065
Accuracy : 78.91%
Precision: 71.87%
Recall   : 94.98%
F1-score : 81.83%
Confusion Matrix
[[6283 3717]
 [ 502 9498]]
Isolation Forest NASA Battery
Run      : 1
Seed     : 42
Threshold: 0.08
Accuracy : 73.52%
Precision: 66.01%
Recall   : 96.99%
F1-score : 78.55%
Confusion Matrix
[[5005 4995]
 [ 301 9699]]

RUN 2/5, seed = 43
Split type: random split by complete source_file cycles
Train normal files: 130
Test normal 